В целом данный код представляет собой адаптированную версию ноутбука по поиску похожих подотрезков для работы с transforms.py и датасетом из src

### 1) Подготовительный этап

In [135]:
from pathlib import Path
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import plotly.graph_objects as go

from src.dataset import Dataset
from src.grapher import plot_time_series
from src.utils.utils import generate_detection_windows
from find_similar import TimeSeriesSubsequenceSearcher
from plots_plotly import add_line, update_layout
from transforms import z_scores_anomaly_transform
from anomaly_detection import DEFAULT_CONFIGURATION, AnomalyDetectionSystem, TimeSeriesWrapper

plt.style.use("seaborn-v0_8-whitegrid")
pd.set_option("display.max_columns", 20)

In [136]:
raw_df = pd.read_csv(dataset.datafiles[0])
print(raw_df['timestamp'].head())

0    1754179200000
1    1754179500000
2    1754179800000
3    1754180100000
4    1754180400000
Name: timestamp, dtype: int64


### 2.1)  Функции обработки введенного датасета

Первая функция преобразует колонку с timestamp из миллисекунд в datetime64
Вторая функция обрезает последние недели временного ряда и возвращает его с кореектными DatetimeIndex

In [137]:
# Ячейка 2: Функции для парсинга и обрезки данных
def parse_timestamp_column(values: pd.Series) -> pd.Series:
    return pd.to_datetime(values, unit='ms', utc=True).dt.tz_convert(None)

def trim_last_weeks(df: pd.DataFrame, weeks: int = 6, value_col: str = None) -> pd.DataFrame:
    timestamps = parse_timestamp_column(df["timestamp"])
    if timestamps.isna().all():
        return pd.DataFrame(index=pd.DatetimeIndex([], name='timestamp'))

    cutoff = timestamps.max() - pd.Timedelta(weeks=weeks)
    mask = timestamps >= cutoff
    filtered_data = df.loc[mask].copy()

    # Определяем колонку со значениями
    if value_col is None:
        numeric_cols = filtered_data.select_dtypes(include=[np.number]).columns.tolist()
        if 'timestamp' in numeric_cols:
            numeric_cols.remove('timestamp')
        if not numeric_cols:
            raise ValueError("Нет числовых колонок для анализа")
        value_col = numeric_cols[0]

    # Создаем одноколоночный DataFrame с правильным индексом
    result_df = filtered_data[['timestamp', value_col]].copy()
    result_df.index = timestamps[mask]
    result_df = result_df.drop(columns=['timestamp'])
    result_df = result_df.sort_index()

    # Удаляем дубликаты индекса, если они есть
    if not result_df.index.is_unique:
        result_df = result_df.groupby(result_df.index).first()

    return result_df

### 2.2) Ввод и обработка датасета

Обрезаем временной ряд до последних 6 недель, после чего сохраняем исходный (raw) и пропущенный через z_scores_anomaly_transform

In [ ]:
dataset = Dataset(".")
series_dict = {}
raw_series_dict = {} 

for csv_path in dataset.datafiles:
    series_id = Path(csv_path).stem
    raw_df = pd.read_csv(csv_path)

    # Обрезаем данные за последние 6 недель, оставляем одну числовую колонку
    ts_data = trim_last_weeks(raw_df, weeks=6)

    raw_series_dict[series_id] = ts_data.copy()

    # Приводим имя колонки к формату, ожидаемому STLDetector
    if len(ts_data.columns) == 1:
        ts_data.columns = ['value_0']
    else:
        ts_data = ts_data.iloc[:, [0]]
        ts_data.columns = ['value_0']

    wrapper = TimeSeriesWrapper(ts_data)
    result_wrapper = z_scores_anomaly_transform(wrapper, AnomalyDetectionSystem(**DEFAULT_CONFIGURATION))

    series_dict[series_id] = result_wrapper.time_series_pd

summary = pd.DataFrame({
    "series_id": list(series_dict.keys()),
    "rows": [len(df) for df in series_dict.values()],
    "columns": [", ".join(df.columns) for df in series_dict.values()],
})
summary

,series_id,rows,columns
0,market_apps_search_request_screen,9216,"value_0, is_anomaly"
1,time_series_1,2530,"value_0, is_anomaly"
2,time_series_0,9504,"value_0, is_anomaly"
3,time_series_2,12097,"value_0, is_anomaly"
4,time_series_3,1152,"value_0, is_anomaly"


Дебаг-вывод для проверки корректной обработки данных

In [139]:
for sid, df in series_dict.items():
    print(sid, "min index:", df.index.min(), "max index:", df.index.max(), "length:", len(df))

market_apps_search_request_screen min index: 2025-08-03 00:00:00 max index: 2025-09-03 23:55:00 length: 9216
time_series_1 min index: 2025-09-18 10:10:00 max index: 2025-09-27 04:55:00 length: 2530
time_series_0 min index: 2025-09-10 13:55:00 max index: 2025-10-13 13:50:00 length: 9504
time_series_2 min index: 2026-03-06 12:00:00 max index: 2026-04-17 12:00:00 length: 12097
time_series_3 min index: 2025-10-13 17:50:00 max index: 2025-10-17 17:45:00 length: 1152


### 2.3) Подготовка данных

In [140]:
searcher = TimeSeriesSubsequenceSearcher(
    freq="30min",
    agg="mean",
    interpolate_limit=3,
    normalize=True,
    exclusion_fraction=0.75,
)

prepared_series = {}
for series_id, df in series_dict.items():
    data_cols = [col for col in df.columns if col != 'is_anomaly']
    if not data_cols:
        raise ValueError(f"Нет числовой колонки в {series_id}")
    val_col = data_cols[0]
    s = df[val_col].resample('30min').mean().interpolate().dropna()
    prepared_series[series_id] = s

# 2. Присваиваем внутренним атрибутам searcher (обходим fit)
searcher.series_ = prepared_series.copy()            # исходные (после ресемплинга)
searcher.prepared_series_ = prepared_series.copy()   # нормализованные (у нас normalize=True)
searcher.freq_ = '30min'

# 3. Устанавливаем размер окна 
searcher.window_size_ = 24

# 4. Вручную строим индексы скользящих окон 
searcher._indices = {}
for sid, s in searcher.prepared_series_.items():
    n = len(s)
    if n >= searcher.window_size_:
        indices = np.arange(n - searcher.window_size_ + 1)
    else:
        indices = np.array([], dtype=int)
    searcher._indices[sid] = indices

# Проверяем результат
prepared_summary = pd.DataFrame({
    "series_id": list(searcher.prepared_series_.keys()),
    "prepared_points": [len(series) for series in searcher.prepared_series_.values()],
    "missing": [int(series.isna().sum()) for series in searcher.prepared_series_.values()],
    "start": [series.index.min() for series in searcher.prepared_series_.values()],
    "end": [series.index.max() for series in searcher.prepared_series_.values()],
})
print(prepared_summary)

                           series_id  prepared_points  missing  \
0  market_apps_search_request_screen             1536        0   
1                      time_series_1              422        0   
2                      time_series_0             1585        0   
3                      time_series_2             2017        0   
4                      time_series_3              193        0   

                start                 end  
0 2025-08-03 00:00:00 2025-09-03 23:30:00  
1 2025-09-18 10:00:00 2025-09-27 04:30:00  
2 2025-09-10 13:30:00 2025-10-13 13:30:00  
3 2026-03-06 12:00:00 2026-04-17 12:00:00  
4 2025-10-13 17:30:00 2025-10-17 17:30:00  


### 3) Поиск похожих последовательностей

In [141]:
source_series_id = "time_series_2" if "time_series_2" in searcher.prepared_series_ else next(iter(searcher.prepared_series_))
query_points = 24

# Обновляем размер окна
searcher.window_size_ = query_points

searcher._indices = {}
for sid, s in searcher.prepared_series_.items():
    n = len(s)
    if n >= searcher.window_size_:
        indices = np.arange(n - searcher.window_size_ + 1)
    else:
        indices = np.array([], dtype=int)
    searcher._indices[sid] = indices

# Берём исходный ряд
source_series = searcher.prepared_series_[source_series_id]
if isinstance(source_series, pd.DataFrame):
    source_series = source_series.iloc[:, 0]
source_series = source_series.dropna()

if len(source_series) < query_points:
    raise ValueError(f"Ряд {source_series_id} слишком короткий: {len(source_series)} < {query_points}")

# Выбираем отрезок
rolling_std = source_series.rolling(query_points).std().to_numpy()
if np.isfinite(rolling_std).any():
    query_end_idx = int(np.nanargmax(rolling_std))
    query_start_idx = query_end_idx - query_points + 1
else:
    query_start_idx = max(0, min(int(len(source_series) * 0.65), len(source_series) - query_points))
    query_end_idx = query_start_idx + query_points - 1

query_start_idx = max(query_start_idx, 0)
query_end_idx = min(query_start_idx + query_points - 1, len(source_series) - 1)
query_start_time = source_series.index[query_start_idx]
query_end_time = source_series.index[query_end_idx]

query_series = source_series.iloc[query_start_idx: query_end_idx + 1]
query_array = query_series.to_numpy().ravel()

print(f"Поиск похожих на отрезок [{query_start_time} — {query_end_time}], длина = {len(query_array)}")

results = searcher.search_by_query_array(
    query_array,
    top_k=50,
    exclude_series_id=source_series_id
)

Поиск похожих на отрезок [2026-04-12 06:30:00 — 2026-04-12 18:00:00], длина = 24


### 4) Визуализация

Градиент цвета от зеленого до красного символизирует похожесть отрезков, где зеленый - максимально похожий, желтый - похойжий, а красный - чем-то напоминающий

In [142]:
distances = results["distance"].values
vmin, vmax = distances.min(), distances.max()

def get_color(distance):
    if vmax != vmin:
        t = (distance - vmin) / (vmax - vmin)
    else:
        t = 0.5
    cmap = plt.cm.get_cmap("RdYlGn_r")
    rgba = cmap(t)
    return f"rgba({int(rgba[0]*255)},{int(rgba[1]*255)},{int(rgba[2]*255)},{rgba[3]})"

# Визуализация
grouped = results.groupby("series_id")
for series_id, group in grouped:
    full_series = searcher.prepared_series_[series_id]
    
    fig = go.Figure()
    add_line(fig, full_series.index, full_series.values, f"Весь ряд {series_id}", "#4d4d4d")
    
    for _, row in group.iterrows():
        segment = searcher.get_segment(row["series_id"], int(row["start_idx"]), int(row["end_idx"]))
        color = get_color(row["distance"])
        fig.add_vrect(
            x0=row["start_time"], x1=row["end_time"],
            fillcolor=color, opacity=0.25,
            layer="below", line_width=0
        )
        add_line(fig, segment.index, segment.values,
                 f"dist={row['distance']:.4f} (idx {row['start_idx']}:{row['end_idx']})",
                 color)
    
    update_layout(fig)
    fig.update_layout(
        title=f"Датасет: {series_id}",
        showlegend=True,
        legend=dict(orientation="v", yanchor="top", y=1, xanchor="left", x=1.02)
    )
    fig.show()

/tmp/ipykernel_11960/3066660887.py:9: MatplotlibDeprecationWarning: The get_cmap function was deprecated in Matplotlib 3.7 and will be removed in 3.11. Use ``matplotlib.colormaps[name]`` or ``matplotlib.colormaps.get_cmap()`` or ``pyplot.get_cmap()`` instead.
  cmap = plt.cm.get_cmap("RdYlGn_r")


/tmp/ipykernel_11960/3066660887.py:9: MatplotlibDeprecationWarning: The get_cmap function was deprecated in Matplotlib 3.7 and will be removed in 3.11. Use ``matplotlib.colormaps[name]`` or ``matplotlib.colormaps.get_cmap()`` or ``pyplot.get_cmap()`` instead.
  cmap = plt.cm.get_cmap("RdYlGn_r")


/tmp/ipykernel_11960/3066660887.py:9: MatplotlibDeprecationWarning: The get_cmap function was deprecated in Matplotlib 3.7 and will be removed in 3.11. Use ``matplotlib.colormaps[name]`` or ``matplotlib.colormaps.get_cmap()`` or ``pyplot.get_cmap()`` instead.
  cmap = plt.cm.get_cmap("RdYlGn_r")


/tmp/ipykernel_11960/3066660887.py:9: MatplotlibDeprecationWarning: The get_cmap function was deprecated in Matplotlib 3.7 and will be removed in 3.11. Use ``matplotlib.colormaps[name]`` or ``matplotlib.colormaps.get_cmap()`` or ``pyplot.get_cmap()`` instead.
  cmap = plt.cm.get_cmap("RdYlGn_r")


In [144]:
source_raw = raw_series_dict[source_series_id]
if isinstance(source_raw, pd.DataFrame):
    source_raw = source_raw.iloc[:, 0] 

fig_raw = go.Figure()
add_line(fig_raw, source_raw.index, source_raw.values, f"Исходный ряд {source_series_id} (без обработки)", "#4d4d4d")

# Используем те же временные границы query (из обработанного ряда)
fig_raw.add_vrect(
    x0=query_start_time, x1=query_end_time,
    fillcolor="rgba(255, 165, 0, 0.3)", layer="below", line_width=0,
    annotation_text="Запрос (query)", annotation_position="top left"
)
# Можно также нарисовать сам query (если в исходных данных есть точно такие же временные метки)
# Для этого нужно извлечь отрезок из исходного ряда по временному интервалу:
query_raw = source_raw.loc[query_start_time:query_end_time] if query_start_time in source_raw.index else None
if query_raw is not None and len(query_raw) > 0:
    add_line(fig_raw, query_raw.index, query_raw.values, "Запрос (исходный)", "orange")
else:
    pass

update_layout(fig_raw)
fig_raw.update_layout(title="Исходные данные (до обработки)", showlegend=True)
fig_raw.show()